In [ ]:
# Install required packages
!pip install pydub mutagen opencv-python numpy pandas matplotlib pillow sqlalchemy pymysql

In [3]:
# Import necessary modules
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sqlalchemy import create_engine  # ADD THIS IMPORT
import os
from PIL import Image, ImageDraw, ImageFont
from pydub import AudioSegment
from mutagen.mp3 import MP3
import cv2
import subprocess

# Your database connection
engine = create_engine('mysql+pymysql://root@localhost:3306/music_development')
data_path = '../data/'


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\User\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\User\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\User\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\Users\User\anaconda3\Lib\site-packages

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: numpy.core.multiarray failed to import

In [ ]:
# Get song data
sql = '''
SELECT S.id, rank, name, artist_id, A.first_name, A.last_name , youtube_code 
FROM songs S
JOIN artists A
ON S.artist_id = A.id
WHERE LOWER(TRIM(name)) LIKE LOWER(TRIM('Just The Way You Are'))
'''
songs = pd.read_sql(sql, engine)
songId = songs['id'].iloc[0]

# Parse lyrics function (unchanged)
def parse_timestamp_lyrics(lyrics_text):
    """Parse lyrics with timestamp format from YouTube transcripts using pipe separator"""
    lines = [line.strip() for line in lyrics_text.split('\n') if line.strip()]
    
    lyrics_with_timing = []
    
    for line in lines:
        if '|' in line:
            timestamp_str, text = line.split('|', 1)
            try:
                timestamp = float(timestamp_str.strip())
                lyrics_with_timing.append({
                    'timestamp': timestamp,
                    'text': text.strip(),
                    'start_time': timestamp,
                    'end_time': None,
                    'duration_seconds': None,
                    'word_count': None,
                    'spaces_count': None,
                    'char_count': None
                })
            except ValueError:
                print(f"⚠️ Skipping line with invalid timestamp: {line}")
                continue
        else:
            print(f"⚠️ Skipping line without Pipe separator: {line}")
    
    lyrics_with_timing.sort(key=lambda x: x['timestamp'])
    
    for i in range(len(lyrics_with_timing)):
        if i < len(lyrics_with_timing) - 1:
            lyrics_with_timing[i]['end_time'] = lyrics_with_timing[i + 1]['timestamp']
        else:
            lyrics_with_timing[i]['end_time'] = lyrics_with_timing[i]['timestamp'] + 5
        
        lyrics_with_timing[i]['duration_seconds'] = lyrics_with_timing[i]['end_time'] - lyrics_with_timing[i]['start_time']
        
        words = [word for word in lyrics_with_timing[i]['text'].split() if word]
        lyrics_with_timing[i]['word_count'] = len(words)
        lyrics_with_timing[i]['spaces_count'] = max(0, lyrics_with_timing[i]['word_count'] - 1)
        lyrics_with_timing[i]['char_count'] = len(lyrics_with_timing[i]['text'])
    
    return lyrics_with_timing

def get_current_lyric(current_time, lyrics_with_timing):
    """Find which lyric should be displayed at current time"""
    for lyric in lyrics_with_timing:
        if lyric['start_time'] <= current_time < lyric['end_time']:
            return lyric['text']
    return None

def get_audio_duration_pydub(audio_path):
    """Get audio duration using pydub"""
    try:
        audio = AudioSegment.from_file(audio_path)
        return len(audio) / 1000.0  # Convert ms to seconds
    except Exception as e:
        print(f"Pydub error: {e}")
        # Fallback to mutagen
        try:
            audio = MP3(audio_path)
            return audio.info.length
        except Exception as e2:
            print(f"Mutagen error: {e2}")
            return None

def create_timestamp_synced_video_pydub(song_id=songId, max_duration=None):
    """Create video using pydub for audio and OpenCV for video"""
    
    try:
        # Get song data
        query = """
        SELECT s.name as song_name, s.location as audio_file,
               l.content as lyrics, a.first_name, a.last_name
        FROM songs s 
        JOIN lyrics l ON s.id = l.song_id 
        JOIN artists a ON s.artist_id = a.id 
        WHERE s.id = %s
        """
        
        df = pd.read_sql(query, engine, params=(song_id,))
        
        if df.empty:
            print(f"❌ No song found with ID: {song_id}")
            return None
            
        song_data = df.iloc[0]
        
        print(f"🎵 Creating TIMESTAMP-SYNCED video for: {song_data['song_name']}")
        
        # Construct file paths
        audio_dir = os.path.join(r"C:\ruby\music\public\uploads\song\location", str(song_id))
        
        if not os.path.exists(audio_dir):
            print(f"❌ Directory doesn't exist: {audio_dir}")
            return None
            
        audio_path = os.path.join(audio_dir, song_data['audio_file'])
        background_image_path = os.path.join(audio_dir, "Folder.jpg")
        
        print(f"🔊 Audio path: {audio_path}")
        print(f"🖼️ Background image: {background_image_path}")
        
        # Check if files exist
        if not os.path.exists(audio_path):
            print(f"❌ Audio file not found at: {audio_path}")
            return None
        
        # Get audio duration using pydub
        full_duration = get_audio_duration_pydub(audio_path)
        if full_duration is None:
            print("❌ Could not get audio duration")
            return None
        
        # Apply duration limit
        if max_duration:
            duration = min(max_duration, full_duration)
            print(f"⏱️ Using LIMITED duration: {duration:.1f}s (max_duration={max_duration}s)")
        else:
            duration = full_duration
            print(f"⏱️ Using FULL duration: {duration:.1f}s")
        
        # Parse timestamp-based lyrics
        lyrics_with_timing = parse_timestamp_lyrics(song_data['lyrics'])
        
        if not lyrics_with_timing:
            print("❌ Could not parse timestamp lyrics")
            return None
        
        print(f"📝 Parsed {len(lyrics_with_timing)} timestamped lyrics lines")
        
        # Filter lyrics to only include those within the limited duration
        if max_duration:
            lyrics_with_timing = [lyric for lyric in lyrics_with_timing if lyric['start_time'] < duration]
            if lyrics_with_timing and lyrics_with_timing[-1]['end_time'] > duration:
                lyrics_with_timing[-1]['end_time'] = duration
        
        print(f"📝 Final timing: {len(lyrics_with_timing)} lyrics lines")
        print(f"⏱️ Video duration: {duration:.1f}s ({duration/60:.1f} minutes)")
        
        # Display parsed timing
        print("\n📋 Parsed Lyrics Timing:")
        print("-" * 80)
        for i, lyric in enumerate(lyrics_with_timing):
            display_text = lyric['text'][:40] + ('...' if len(lyric['text']) > 40 else '')
            print(f"{i+1:2d}. {lyric['start_time']:5.1f}s - {lyric['end_time']:5.1f}s: "
                  f"{display_text:<43} | "
                  f"{lyric['duration_seconds']:4.1f}s | "
                  f"{lyric['word_count']:2d} | "
                  f"{lyric['spaces_count']:2d} | "
                  f"{lyric['char_count']:3d}")
        
        # Video settings
        fps = 24
        width, height = 640, 480
        
        # Prepare output file
        output_dir = '../data/videos'
        os.makedirs(output_dir, exist_ok=True)
        
        clean_artist_name = f"{song_data['first_name']} {song_data['last_name']}"
        video_file = os.path.join(output_dir, f"{song_data['song_name']}_{clean_artist_name}.mp4")
        temp_video_file = os.path.join(output_dir, f"{song_data['song_name']}_temp.mp4")
        
        # Create video frames
        print("🎬 Creating video frames...")
        
        # Initialize video writer
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(temp_video_file, fourcc, fps, (width, height))
        
        # Load background image if exists
        if os.path.exists(background_image_path):
            bg_image = Image.open(background_image_path)
            bg_image = bg_image.resize((width, height), Image.Resampling.LANCZOS)
            bg_np = np.array(bg_image)
        else:
            bg_np = np.full((height, width, 3), [40, 40, 80], dtype=np.uint8)
        
        # Create frames
        total_frames = int(duration * fps)
        
        for frame_num in range(total_frames):
            current_time = frame_num / fps
            
            # Create frame
            frame = bg_np.copy()
            
            # Get current lyric
            current_line = get_current_lyric(current_time, lyrics_with_timing)
            
            if current_line:
                # Convert to PIL for text drawing
                pil_img = Image.fromarray(frame)
                draw = ImageDraw.Draw(pil_img)
                
                # Load font
                try:
                    font = ImageFont.truetype("arial.ttf", 32)
                except:
                    try:
                        font = ImageFont.truetype("C:/Windows/Fonts/arial.ttf", 32)
                    except:
                        font = ImageDraw.ImageFont.load_default()
                
                # Calculate text position
                try:
                    bbox = draw.textbbox((0, 0), current_line, font=font)
                except AttributeError:
                    bbox = draw.textsize(current_line, font=font)
                    bbox = (0, 0, bbox[0], bbox[1])
                
                text_width = bbox[2] - bbox[0]
                text_height = bbox[3] - bbox[1]
                x = (width - text_width) // 2
                y = height - 50
                
                # Draw text
                draw.text((x, y), current_line, font=font, fill=(255, 255, 255))
                
                # Convert back to OpenCV format
                frame = np.array(pil_img)
            
            # Write frame
            out.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
            
            # Progress indicator
            if frame_num % (fps * 5) == 0:  # Every 5 seconds
                print(f"  Progress: {current_time:.1f}s / {duration:.1f}s")
        
        out.release()
        print("✅ Video frames created")
        
        # Combine video and audio using ffmpeg
        print("🔊 Adding audio to video...")
        
        # Use ffmpeg to combine video and audio
        ffmpeg_cmd = [
            'ffmpeg',
            '-i', temp_video_file,
            '-i', audio_path,
            '-c:v', 'copy',
            '-c:a', 'aac',
            '-strict', 'experimental',
            '-map', '0:v:0',
            '-map', '1:a:0',
            '-shortest',
            video_file
        ]
        
        try:
            # Check if ffmpeg is available
            subprocess.run(['ffmpeg', '-version'], capture_output=True, check=True)
            
            # Run ffmpeg command
            result = subprocess.run(ffmpeg_cmd, capture_output=True, text=True)
            if result.returncode == 0:
                print(f"✅ Video with audio created: {video_file}")
                
                # Get file size
                if os.path.exists(video_file):
                    file_size = os.path.getsize(video_file) / (1024*1024)
                    print(f"📊 File size: {file_size:.1f} MB")
                    
                    # Clean up temp file
                    if os.path.exists(temp_video_file):
                        os.remove(temp_video_file)
                    
                    return video_file
                else:
                    print("❌ Output video file not created")
            else:
                print(f"❌ FFmpeg error: {result.stderr}")
                
        except (subprocess.CalledProcessError, FileNotFoundError):
            print("⚠️ FFmpeg not found. Creating video without audio...")
            # Rename temp file to final video file
            if os.path.exists(temp_video_file):
                os.rename(temp_video_file, video_file)
                print(f"✅ Video created (no audio): {video_file}")
                return video_file
        
        return None
        
    except Exception as e:
        print(f"❌ Video creation error: {e}")
        import traceback
        traceback.print_exc()
        return None

# Run the function
if __name__ == "__main__":
    print("=" * 70)
    print("🎬 CREATING TIMESTAMP-SYNCED VIDEO (PYDUB VERSION)")
    print("=" * 70)
    
    result = create_timestamp_synced_video_pydub(song_id=songId, max_duration=None)
    
    if result:
        print(f"\n🎉 SUCCESS! Video created: {result}")
        print("\n✨ Features:")
        print("   ✅ YouTube transcript timestamp parsing")
        print("   ✅ Pydub for audio processing (no moviepy needed)")
        print("   ✅ OpenCV for video creation")
        print("   ✅ Perfect synchronization with original timestamps")
    else:
        print("\n❌ Video creation failed")